In [1]:
using Revise
using InteractiveUtils

const PATH_CLUSTERING_JL = "functions/cluster_analysis.jl"
# const PATH_PDF_EXTRACT_JL = "functions/qog_pdf_extract.jl"
# const PATH_METADATA_ENHANCE_JL = "functions/qog_metadata_join.jl"

# Print a summary of all dataframes that are current loaded in Main
function dataframe_summaries(mod=Main)
    for n in names(mod)
        x = getfield(mod, n)
        if x isa AbstractDataFrame
            println("=== DataFrame: ", n, " ===")
            println(summary(x))
            println()
        end
    end
end

includet(PATH_CLUSTERING_JL)

In [2]:
df, meta = load_dataframes();

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [3]:
dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2013 DataFrame

=== DataFrame: meta ===
2010×16 DataFrame



In [4]:
country_features_df, audit = build_country_features_df(df, meta);

In [5]:
size(audit)

(2009, 5)

In [6]:
size(country_features_df)

(200, 22101)

In [7]:
check_country_features_qa(df, country_features_df);


>>> Step 1 QA: country_features_df
    Unique ident_ccode in df: 200
    Rows in country_features_df: 200
    Row count match: PASS
    For a known global slug with good coverage, missrates should be mostly low in later periods.
    For regional-only slugs, missrates should not penalize countries outside region (applicable_n==0 → missing).


In [9]:
kept, rejected, report = filter_slugs_step2(country_features_df, meta; df_panel=df);

┌ Warning: require_periods != length(periods); using require_periods=1 and periods=4
└ @ Main ~/work/functions/cluster_analysis.jl:667



Step 2 — Slug Filtering Diagnostics
Total candidate slugs: 2009
Kept slugs:            0
Rejected slugs:        2009
Keep rate:             0.0 %

Rejection reasons:
3×2 DataFrame
 Row │ reason           count 
     │ String           Int64 
─────┼────────────────────────
   1 │ low_coverage      1993
   2 │ identifier_like     14
   3 │ non_numeric          2

Coverage statistics (kept slugs):

Example kept slugs:


Example rejected slugs:
aid_cpnc, aid_cpsc, aid_crnc, aid_crnio, aid_crsc, aid_crsio, aii_acc, aii_aio, aii_cilser, aii_elec



In [20]:
run_cluster_samples()


  cluster_analysis.jl — Function intent and usage (Step 1: country-level features)

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the clustering metadata in one call.
│  USE WHEN: You need both df and meta_df for building country features.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_CLUSTER_INPUT (qog_metadata_plus2.csv)
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ build_country_features_df(df, meta_df; periods, min_obs_per_period, min_obs_for_vol, include_audit)
│  INTENT: Build country-level feature table (one row per ident_ccode) with period means,
│          delta_total, recent_change, volatility, and period missrates per slug.
│  USE WHEN: Step 1 of clustering prep — you need country-level summary for clustering.
│
│  ARGUMENTS:
│    df::DataFrame — panel with ident_ccode, ident_year, ggis_region, plus slug co